In [ ]:
import boto3, sagemaker
import os
ACCOUNT_ID = os.getenv("AWS_ACCOUNT_ID") #replace with your aws account ID
REGION = os.getenv("AWS_REGION", "us-east-1") 
ROLE_ARN   = f"arn:aws:iam::{ACCOUNT_ID}:role/ThreatDetectionSageMakerRole"

sess      = sagemaker.Session()
sm_client = boto3.client("sagemaker", region_name=REGION)

packages  = sm_client.list_model_packages(
    ModelPackageGroupName = "ThreatDetectionModels",
    ModelApprovalStatus   = "Approved",
    SortBy                = "CreationTime",
    SortOrder             = "Descending"
)
latest_arn = packages["ModelPackageSummaryList"][0]["ModelPackageArn"]
print(f"Deploying model: {latest_arn}")

model = sagemaker.ModelPackage(
    role              = ROLE_ARN,
    model_package_arn = latest_arn,
    sagemaker_session = sess
)

predictor = model.deploy(
    initial_instance_count = 1,
    instance_type          = "ml.m5.large",
    endpoint_name          = "threat-detection-endpoint",
    serializer             = sagemaker.serializers.CSVSerializer(),
    deserializer           = sagemaker.deserializers.JSONDeserializer()
)
print("Endpoint live:", predictor.endpoint_name)

In [ ]:
#Test
runtime  = boto3.client("sagemaker-runtime", region_name="us-east-1")
ENDPOINT = "threat-detection-endpoint"

# Sample network event: src_ip_int, dst_ip_int, src_port, dst_port,
# protocol, bytes_sent, bytes_recv, duration_ms, packet_count, flag, byte_ratio
sample = "167772160,134744072,54321,443,0,15000,3200,120,45,1,4.69"

response = runtime.invoke_endpoint(
    EndpointName = ENDPOINT,
    ContentType  = "text/csv",
    Body         = sample
)

score = float(response["Body"].read().decode("utf-8"))
label = "SUSPICIOUS" if score >= 0.5 else "NORMAL"
print(f"Threat score: {score:.4f}")
print(f"Classification: {label}")